# M2 T3 Practical — Gaussian Mixture Models

Revision: 15 September 2026

## Learning outcomes

- Interpret fitted weights, means and standard deviations.
- Find and explain uncertain model-based responsibilities.
- Distinguish density from interval probability.
- Compare BIC evidence and model assumptions.
- Implement a guided E-step and M-step and investigate EM initialisation.
- Implement and validate the modelling or evaluation code required in this activity.


## How to work and submit

Plan for a two-hour session: about 90 minutes of core work and 30 minutes for discussion, debugging and checking. Timings beside questions are estimates, not deadlines. Setup and plotting support are supplied; focus on the modelling ideas.

Run cells in order. Replace `...` in TODO cells and write in each answer cell. Make a prediction **before** running an experiment; keep it even if the result surprises you, then explain the difference. If a cell fails, ask about the first error and retain your genuine attempt.

For every explanation, cite a number, observation or intermediate value from your own run, explain what it means, and justify your conclusion. Concise supported answers are enough. Different justified choices may be valid. Save your notebook with outputs and written answers. Supplied code alone is not a completed response.

The five core questions are assessed using the course's 50% completion/genuine attempt and 50% demonstrated understanding criteria. Optional extensions are not required. No new package installation or external dataset download is needed in the course Python environment (NumPy, pandas, matplotlib and scikit-learn).

## Preparation and links to the topic

Review [M2 T3 overview](https://learn.adelaide.edu.au/courses/30362/pages/module-2-topic-3-overview), [Mixture of Gaussians — Math](https://learn.adelaide.edu.au/courses/30362/pages/mixture-of-gaussians-math) and [Evaluating Unsupervised Learning](https://learn.adelaide.edu.au/courses/30362/pages/evaluating-unsupervised-learning). Q1–3 connect density, responsibilities and model-count evidence; Q4–5 apply the taught E/M formulas and investigate initialisation. No convergence proof or BIC derivation is required.

The notebook uses one dimension: covariance reduces to variance, and standard deviation is its square root. Responsibilities are conditional component probabilities, not observed class labels. The simple EM code is intended for these supplied data. For extreme values, multidimensional data or nearly empty components, a robust implementation needs additional numerical safeguards.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.mixture import GaussianMixture

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
x = np.concatenate([
    rng.normal(-2.2, 0.65, 210),
    rng.normal(2.0, 1.05, 140),
]).reshape(-1, 1)

plt.hist(x[:,0], bins=35, density=True, alpha=0.35, edgecolor="white")
plt.xlabel("x"); plt.ylabel("density")
plt.title("Observed values: component labels are hidden")
plt.show()

## Question 1 — Interpret a fitted mixture (15 minutes)

Write the code to construct and fit a two-component GaussianMixture with n_init=10 and the supplied random state. Build a table of fitted weights, means and standard deviations, sorted by mean. Useful attributes: weights_, means_, covariances_; in this one-dimensional task, flatten the arrays and take the square root of variances. Report these three parameter types and explain why component numbers are arbitrary. Keep your explanation to one short paragraph.

In [ ]:
# TODO: construct and fit gmm, then build summary as specified above.
# Keep component weights/means/standard deviations aligned when sorting.
gmm = GaussianMixture(
    n_components=2,
    n_init=10,
    random_state=RANDOM_STATE,
)
gmm.fit(x)
# Write the fitting and table-building statements here.

order = np.argsort(gmm.means_.flatten())
summary = pd.DataFrame({
    "weight": gmm.weights_.flatten()[order],
    "mean": gmm.means_.flatten()[order],
    "sd": np.sqrt(gmm.covariances_.reshape(-1))[order]
})
display(summary.round(3))


### Write your answer — Question 1

The fitted mixture has weights of about 0.596 and 0.404, means of about -2.237 and 1.998, and standard deviations of about 0.564 and 1.123. The weights add up to 1 and show how much of the model is assigned to each component. Around 59.6% is assigned to the left component before observing a specific x. The means show where the centres of the components are, while the standard deviations show how spread out the x values are. The right component has a standard deviation that is about twice as large, meaning it is roughly twice as wide. The component numbers are arbitrary because a Gaussian mixture does not have fixed class labels, so swapping component 0 and component 1 would represent the same fitted mixture.

## Question 2 — Find uncertain observations (15 minutes)

Compute responsibilities for the supplied probe grid. Find the most uncertain row programmatically by minimising the largest responsibility in each row (np.max with axis=1, then np.argmin). Also identify the most confident row. For two components, the most uncertain row has responsibilities closest to [0.5,0.5]. Report both responsibility vectors, explain why weights and component densities matter, and give one use for retaining soft membership. Do not hard-code a row index.

In [ ]:
probe_grid = np.linspace(-2.5, 2.5, 21).reshape(-1, 1)
# TODO: obtain the (n, 2) responsibility matrix, reduce across components,
# and select the uncertain and confident indices without hard-coding them.
probe_r = gmm.predict_proba(probe_grid)
confidence = probe_r.max(axis=1)
chosen_row = np.argmin(confidence)
confident_row = np.argmax(confidence)

for label, index in [('most uncertain', chosen_row), ('most confident', confident_row)]:
    print(label, probe_grid[index], probe_r[index])
assert np.allclose(probe_r.sum(axis=1), 1)
assert confidence[chosen_row] <= confidence[confident_row]


### Write your answer — Question 2

For x = -0.75, the responsibilities are approximately [0.6435, 0.3565], making it the most balanced and therefore the most uncertain point. In comparison, x = 2.5 has [0.997, 0.0003], which is much more confident. Since the grid is finite, the selected row does not have to be exactly [0.5, 0.5] or be the most uncertain point across the whole real line.

The distance to the mean is only one part of the density. The standard deviation affects both the normalising height and the scaled distance, while the mixture weights affect the prior likelihood of each component. Because of this, the crossover point does not necessarily have to be the midpoint between the two means.

The responsibilities sum to one because they represent how the membership is distributed between the fitted components. This does not mean that the components have been confirmed as real customer classes. K-means would instead assign each point to the nearest centroid and give a single hard label, without a responsibility vector. Taking the component with the largest GMM responsibility would also give a hard label, but it does not necessarily have to agree with K-means. Soft membership is useful when customer behaviours overlap and we want to keep some uncertainty instead of forcing each customer into a single category.


## Question 3 — Density and model-count evidence (20 minutes)

Run the supplied density plot. Write a loop fitting component counts 1 through 5, with n_init=10 and the same random state. Collect each count, BIC and AIC in a results table and select the minimum-BIC row programmatically. Briefly explain how the weighted densities form the mixture, why density is not point probability, and why the BIC-preferred count is not proof of natural groups. Cite the selected BIC and one alternative.

In [ ]:
grid = np.linspace(x.min()-1, x.max()+1, 500).reshape(-1,1)
mixture_density = np.exp(gmm.score_samples(grid))
grid_resp = gmm.predict_proba(grid)
weighted_components = grid_resp * mixture_density[:,None]

plt.hist(x[:,0], bins=35, density=True, alpha=0.25, edgecolor="white", label="data")
plt.plot(grid[:,0], mixture_density, color="black", linewidth=2, label="mixture")
for k in range(2):
    plt.plot(grid[:,0], weighted_components[:,k], "--", label=f"weighted component {k}")
plt.xlabel("x"); plt.ylabel("density"); plt.legend(); plt.show()

# TODO: write the model-comparison loop and create criteria with columns
# components, BIC, AIC. Use candidate.bic(x) and candidate.aic(x).
rows = []
for k in range(1, 6):
    candidate = GaussianMixture(n_components=k, n_init=10, random_state=RANDOM_STATE)
    candidate.fit(x)
    rows.append({
        "components": k,
        "BIC": candidate.bic(x),
        "AIC": candidate.aic(x)
    })

criteria = pd.DataFrame(rows)
best_components = criteria.loc[criteria['BIC'].idxmin(), 'components']
display(criteria.round(1))
print('BIC-preferred component count:', best_components)


### Write your answer — Question 3
Each dashed curve represents weight_k × component_density_k. Adding the dashed curves together at each point gives the black mixture curve. In the supplied code, multiplying the responsibility by the mixture density gives back the weighted component density by rearranging the responsibility formula. The total area under each dashed curve is its mixture weight, while the total area under the black curve is 1.
For continuous x, probability is area over an interval: P(a <= X <= b) = integral_ab p(x) dx.
The probability of getting one exact x value is zero, even if the density is high at that point. Density is measured in inverse-x units, so it is not a percentage.

For the model count, the BIC is about 1276.3 for two components, 1293.0 for three components and 1569.5 for one component. Since lower BIC is better, the two-component model is preferred from these options. The three-component model is about 16.7 BIC units worse than the two-component model.

BIC is lower when the balance between the model's fit and its number of parameters is better. AIC is also printed in the results, but BIC is used here to choose the model, so no AIC derivation is needed.

A Gaussian mixture can use multiple components to approximate a population that is not normally distributed. Therefore, choosing two components as the best model under BIC does not prove that there are two natural groups in the data. The result can still depend on the data, covariance assumptions and the fitted solution.

In [ ]:
def normal_pdf(values, mean, sd):
    return np.exp(-0.5*((values-mean)/sd)**2)/(np.sqrt(2*np.pi)*sd)
values=x[:,0]
initial=(np.array([0.5,0.5]),np.array([-1.5,1.5]),np.array([1.2,1.2]))

## Question 4 — Build one EM iteration (25 minutes)

First calculate by hand the responsibilities from weighted densities [0.12,0.08]. Complete and run e_step before m_step. Then implement the weighted parameter updates. Shapes: responsibilities (n,2), values[:,None] (n,1), component counts (2,). The formulas are weights=N_k/n, mean_k=sum(r_ik*x_i)/N_k, variance_k=sum(r_ik*(x_i-new_mean_k)^2)/N_k. Explain what is fixed in each step and use the row-sum and weight-sum checks to validate your implementation.

In [ ]:
def e_step(values, weights, means, sds):
    weighted=np.column_stack([weights[k]*normal_pdf(values,means[k],sds[k]) for k in range(2)])
    # TODO: normalise across components, retaining the row dimension.
    return weighted / weighted.sum(axis=1, keepdims=True)
def m_step(values, responsibilities):
    counts=responsibilities.sum(axis=0)
    # TODO: weights = effective counts / number of observations.
    weights = counts / len(values)
    # TODO: weighted means; values[:,None] has shape (n,1).
    means = (responsibilities * values[:, None]).sum(axis=0) / counts
    # TODO: weighted squared deviations from the UPDATED means, divided by counts.
    variances = (responsibilities * (values[:, None] - means)**2).sum(axis=0) / counts
    return weights, means, np.sqrt(np.maximum(variances,1e-12))
r=e_step(values,*initial)
assert np.allclose(r.sum(axis=1),1)
print('First responsibilities:',r[:3])
one_update=m_step(values,r)
print('Updated weights, means, sds:',one_update)
assert np.isclose(one_update[0].sum(),1)
assert np.all(one_update[0]>=0) and np.all(one_update[2]>0)
print("Responsibility and parameter checks passed.")

### Write your answer — Question 4

For the hand normalisation, the total weighted density is 0.12 + 0.08 = 0.20, so the responsibilities are 0.12 / 0.20 = 0.6 and 0.08 / 0.20 = 0.4. Dividing by the sum within each row is the E-step. axis=1, keepdims=True keeps the shape as (n,1), which allows each row's two values to use its own denominator.

For the M-step, sum the responsibilities down each column to get the fractional counts N_k. These counts are then divided by n = 350 to get the new weights. For the new mean, multiply each x by its responsibility, sum the results and divide by N_k.

The new standard deviation is calculated by weighting the squared deviations from the new mean by the responsibilities, dividing by N_k and then taking the square root. The divisor is N_k rather than N_k - 1 because this is a maximum-likelihood update.

The reference first update gives weights [0.616960, 0.383040], means [-2.115770, 2.033414] and standard deviations [0.811798, 1.190269]. The first observation has a responsibility of approximately [0.984793, 0.015207]. During the E-step, the model parameters stay fixed, while during the M-step, the calculated responsibilities stay fixed.

The checks should confirm that every responsibility row sums to 1, the weights sum to 1 and the standard deviations are positive. These checks can catch common mistakes, but they do not prove that every formula is correct.

The small variance floor prevents division by zero. This implementation is mainly for the supplied well-behaved data and is not intended to be a numerically robust general-purpose GMM implementation.


In [ ]:
def run_em(start, steps=8):
    parameters=tuple(a.copy() for a in start);history=[]
    for iteration in range(steps+1):
        w,mu,sd=parameters
        density=sum(w[k]*normal_pdf(values,mu[k],sd[k]) for k in range(2))
        history.append([iteration,np.log(np.maximum(density,1e-300)).sum(),*mu])
        if iteration<steps: parameters=m_step(values,e_step(values,*parameters))
    return parameters,pd.DataFrame(history,columns=['iteration','log_likelihood','mean_0','mean_1'])
def parameter_table(parameters):
    w,mu,sd=parameters
    return pd.DataFrame({'weight':w,'mean':mu,'sd':sd}).sort_values('mean')

## Question 5 — Test initialisation and diagnose EM (15 minutes)

Before running, predict whether two identical starting components will separate under these deterministic updates. Compare start A from Q4 with start B below. Report likelihood changes and final parameters after sorting by mean. Compare with the fitted scikit-learn model from Q1. Explain whether rising/non-decreasing likelihood guarantees a globally best fit and why exact parameter equality is unnecessary.

In [ ]:
start_b=(np.array([0.5,0.5]),np.array([0.,0.]),np.array([1.2,1.2]))
for label,start in [("A",initial),("B",start_b)]:
    params,history=run_em(start)
    print(label);display(history.round(3));display(parameter_table(params).round(3))
display(summary.round(3))

### Write your answer — Question 5

Start A separates the two components. Its total log-likelihood increases from -745.050 to -623.503 after eight updates, which is an improvement of about 121.547. Its final weights are [0.596, 0.404], means [-2.238, 1.995] and standard deviations [0.563, 1.126]. These are close to the Q1 results after ordering the components by their means.

Start B cannot break its exact symmetry. With equal weights, means and standard deviations, both components have identical weighted densities, so the responsibility is 0.5 for both components at every observation. Both means update to the same overall mean, about -0.526, both standard deviations become about 2.240, and both weights remain at 0.5.

The log-likelihood increases from -1028.912 to -778.898 and then stays there. This is still worse than Start A even though the likelihood initially increases.

The non-decreasing likelihood property of EM with exact updates does not guarantee that it will find the globally best configuration. In this case, the symmetry leads to an inferior stationary solution. Using several asymmetric starting points can help show the sensitivity to initialisation, but it still does not prove global optimality.

The eight iterations, stopping tolerances and variance regularisation can also cause small differences compared with scikit-learn. When comparing the results, compare the densities and parameters after aligning the component labels rather than expecting the component numbers to be identical.


## Optional extensions

Complete these only after the core. Each is worth up to 5 bonus marks under the course policy; omitting either loses no core marks.

- Try a third asymmetric initialisation and compare convergence and final likelihood.
- Fit bootstrap samples with the supplied GaussianMixture API and compare means after sorting; describe sampling instability.